<a href="https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://raw.githubusercontent.com/sokrypton/ColabFold/main/.github/ColabFold_Marv_Logo_Small.png" height="200" align="right" style="height:240px">

## ColabFold v1.5.5: AlphaFold2 using MMseqs2

Easy to use protein structure and complex prediction using [AlphaFold2](https://www.nature.com/articles/s41586-021-03819-2) and [Alphafold2-multimer](https://www.biorxiv.org/content/10.1101/2021.10.04.463034v1). Sequence alignments/templates are generated through [MMseqs2](mmseqs.com) and [HHsearch](https://github.com/soedinglab/hh-suite). For more details, see <a href="#Instructions">bottom</a> of the notebook, checkout the [ColabFold GitHub](https://github.com/sokrypton/ColabFold) and [Nature Protocols](https://www.nature.com/articles/s41596-024-01060-5).

Old versions: [v1.4](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.4.0/AlphaFold2.ipynb), [v1.5.1](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.1/AlphaFold2.ipynb), [v1.5.2](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.2/AlphaFold2.ipynb), [v1.5.3-patch](https://colab.research.google.com/github/sokrypton/ColabFold/blob/56c72044c7d51a311ca99b953a71e552fdc042e1/AlphaFold2.ipynb)

[Mirdita M, Schütze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M. ColabFold: Making protein folding accessible to all.
*Nature Methods*, 2022](https://www.nature.com/articles/s41592-022-01488-1)

In [ ]:
#@title Input protein sequence(s)
try:
  from google.colab import files
except ImportError:
  files = None

!pip install -q ezinput==0.0.12

import os
import re
import hashlib

from ezinput import EZInput
from IPython.display import display, Markdown, clear_output

from sys import version_info
python_version = f"{version_info.major}.{version_info.minor}"

def add_hash(x,y):
  return x+"_"+hashlib.sha1(y.encode()).hexdigest()[:5]

# check if directory with jobname exists
def check(folder):
  if os.path.exists(folder):
    return False
  else:
    return True

input_protein_gui = EZInput(title="AlphaFold2_advanced_input")

input_protein_gui.add_text(tag="query_sequence",
			description="Input protein sequence(s):",
			placeholder="PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK",
			remember_value=True)
input_protein_gui.add_HTML(tag="temp_output",
			value=" - Use <code>:</code> to specify inter-protein chainbreaks for <b>modeling complexes</b> (supports homo- and hetro-oligomers). For example <b>PI...SK:PI...SK</b> for a homodimer")
input_protein_gui.add_text(tag="jobname",
			description="Job name:",
			placeholder="test",
			remember_value=True)
# number of models to use
input_protein_gui.add_dropdown(tag="num_relax",
				description="Number of models to relax using Amber:",
				options=[0, 1, 5],
				value=0,
				remember_value=True)

input_protein_gui.add_HTML(tag="temp_output_2",
				value=" - specify how many of the top ranked structures to relax using amber")
input_protein_gui.add_dropdown(tag="template_mode",
				description="Template mode:",
				options=["none", "pdb100","custom"],
				value="none",
				remember_value=True)
input_protein_gui.add_HTML(tag="temp_output_3",
				value="- <code>none</code> = no template information is used. <code>pdb100</code> = detect templates in pdb100 (see [notes](#pdb100)). <code>custom</code> - upload and search own templates (PDB or mmCIF format, see [notes](#custom_templates))")

use_amber, query_sequence, jobname, num_relax, template_mode = None, None, None, None, None
use_templates, custom_template_path, queries_path = None, None, None
def run_cell(_):
	global use_amber, query_sequence, jobname, num_relax, template_mode, use_templates, custom_template_path, queries_path

	query_sequence = input_protein_gui["query_sequence"].value
	jobname = input_protein_gui["jobname"].value
	num_relax = input_protein_gui["num_relax"].value
	template_mode = input_protein_gui["template_mode"].value

	with input_protein_gui["output"]:
		clear_output()

		use_amber = num_relax > 0

		# remove whitespaces
		query_sequence = "".join(query_sequence.split())

		basejobname = "".join(jobname.split())
		basejobname = re.sub(r'\W+', '', basejobname)
		jobname = add_hash(basejobname, query_sequence)

		if not check(jobname):
			n = 0
			while not check(f"{jobname}_{n}"): n += 1
			jobname = f"{jobname}_{n}"

		# make directory to save results
		os.makedirs(jobname, exist_ok=True)

		# save queries
		queries_path = os.path.join(jobname, f"{jobname}.csv")
		with open(queries_path, "w") as text_file:
			text_file.write(f"id,sequence\n{jobname},{query_sequence}")

		if template_mode == "pdb100":
			use_templates = True
			custom_template_path = None
		elif template_mode == "custom":
			custom_template_path = os.path.join(jobname,f"template")
			os.makedirs(custom_template_path, exist_ok=True)
			uploaded = files.upload()
			use_templates = True
			for fn in uploaded.keys():
				os.rename(fn,os.path.join(custom_template_path,fn))
		else:
			custom_template_path = None
			use_templates = False

		print("jobname",jobname)
		print("sequence",query_sequence)
		print("length",len(query_sequence.replace(":","")))

input_protein_gui.add_callback("run_cell", func=run_cell, values=input_protein_gui.elements, button_text="Submit")
input_protein_gui.add_output(tag="output")

# Display the GUI
input_protein_gui.show()

# Save settings
input_protein_gui._save_settings()

In [ ]:
#@title Install dependencies
# %%time
import os
USE_AMBER = use_amber
USE_TEMPLATES = use_templates
PYTHON_VERSION = python_version

if not os.path.isfile("COLABFOLD_READY"):
  print("installing colabfold...")
  os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
  if os.environ.get('TPU_NAME', False) != False:
    os.system("pip uninstall -y jax jaxlib")
    os.system("pip install --no-warn-conflicts --upgrade dm-haiku==0.0.10 'jax[cuda12_pip]'==0.3.25 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
  # hack to fix TF crash
  os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so")
  os.system("touch COLABFOLD_READY")

if USE_AMBER or USE_TEMPLATES:
  if not os.path.isfile("CONDA_READY"):
    print("installing conda...")
    os.system("wget -qnc https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh")
    os.system("bash Miniforge3-Linux-x86_64.sh -bfp /usr/local")
    os.system("mamba config --set auto_update_conda false")
    os.system("touch CONDA_READY")

if USE_TEMPLATES and not os.path.isfile("HH_READY") and USE_AMBER and not os.path.isfile("AMBER_READY"):
  print("installing hhsuite and amber...")
  os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
  os.system("touch HH_READY")
  os.system("touch AMBER_READY")
else:
  if USE_TEMPLATES and not os.path.isfile("HH_READY"):
    print("installing hhsuite...")
    os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 python='{PYTHON_VERSION}'")
    os.system("touch HH_READY")
  if USE_AMBER and not os.path.isfile("AMBER_READY"):
    print("installing amber...")
    os.system(f"mamba install -y -c conda-forge openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
    os.system("touch AMBER_READY")

In [ ]:

#@title MSA options (custom MSA upload, single sequence, pairing mode)

msa_gui = EZInput(title="AlphaFold2_msa_options")

msa_gui.add_HTML(tag="temp_output_0",
				value="<h3>MSA options (custom MSA upload, single sequence, pairing mode)</h3>")
msa_gui.add_dropdown("msa_mode",
				description="MSA mode:",
				options=["mmseqs2_uniref_env", "mmseqs2_uniref","single_sequence","custom"],
				value="mmseqs2_uniref_env",
				remember_value=True)
msa_gui.add_dropdown("pair_mode",
				description="Pairing mode:",
				options=["unpaired_paired","paired","unpaired"],
				value="unpaired_paired",
				remember_value=True)

msa_gui.add_HTML(tag="temp_output_1",
				value="- <code>unpaired_paired</code> = pair sequences from same species + unpaired MSA, <code>unpaired</code> = separate MSA for each chain, <code>paired</code> - only use paired sequences.")

msa_mode, pair_mode, query_sequence, a3m_file = None, None, None, None

def run_cell_msa(_):
	global msa_mode, pair_mode, queries_path, a3m_file, query_sequence
	msa_mode = msa_gui["msa_mode"].value
	pair_mode = msa_gui["pair_mode"].value

	with msa_gui["output"]:
		clear_output()
		# decide which a3m to use
		if "mmseqs2" in msa_mode:
			a3m_file = os.path.join(jobname,f"{jobname}.a3m")

		elif msa_mode == "custom":
			a3m_file = os.path.join(jobname,f"{jobname}.custom.a3m")
			if not os.path.isfile(a3m_file):
				custom_msa_dict = files.upload()
				custom_msa = list(custom_msa_dict.keys())[0]
				header = 0
				import fileinput
				for line in fileinput.FileInput(custom_msa,inplace=1):
					if line.startswith(">"):
						header = header + 1
					if not line.rstrip():
						continue
					if line.startswith(">") == False and header == 1:
						query_sequence = line.rstrip()
					print(line, end='')

				os.rename(custom_msa, a3m_file)
				queries_path=a3m_file
				print(f"moving {custom_msa} to {a3m_file}")

		else:
			a3m_file = os.path.join(jobname,f"{jobname}.single_sequence.a3m")
			with open(a3m_file, "w") as text_file:
				text_file.write(">1\n%s" % query_sequence)

		print("MSA mode:",msa_mode)
		print("Pairing mode:",pair_mode)
		print("Using MSA file:",a3m_file)

msa_gui.add_callback("run_cell_msa", func=run_cell_msa, values=msa_gui.elements, button_text="Submit")
msa_gui.add_output(tag="output")

# Display the GUI
msa_gui.show()

# Save settings
msa_gui._save_settings()

In [ ]:
#@title Advanced settings

advanced_settings_gui = EZInput(title="AlphaFold2_advanced_settings")

advanced_settings_gui.add_HTML(tag="temp_output_adv",
				value="<h3>Advanced settings</h3>")
advanced_settings_gui.add_dropdown(tag="model_type",
				description="Model type:",
				options=["auto", "alphafold2_ptm", "alphafold2_multimer_v1", "alphafold2_multimer_v2", "alphafold2_multimer_v3", "deepfold_v1", "alphafold2"],
				remember_value=True)
advanced_settings_gui.add_HTML(tag="temp_output_adv_1",
				value="- if <code>auto</code> selected, will use <code>alphafold2_ptm</code> for monomer prediction and <code>alphafold2_multimer_v3</code> for complex prediction.<br>Any of the mode_types can be used (regardless if input is monomer or complex).")

advanced_settings_gui.add_dropdown(tag="num_recycles",
				description="Number of recycles:",
				options=["auto", "0", "1", "3", "6", "12", "24", "48"],
				remember_value=True)
advanced_settings_gui.add_HTML(tag="temp_output_adv_2",
				value= "- if <code>auto</code> selected, will use <code>num_recycles=20</code> if <code>model_type=alphafold2_multimer_v3</code>, else <code>num_recycles=3</code> .")
advanced_settings_gui.add_dropdown(tag="recycle_early_stop_tolerance",
				description="Recycle early stop tolerance:",
				options=["auto", "0.0", "0.5", "1.0"],
				remember_value=True)
advanced_settings_gui.add_HTML(tag="temp_output_adv_3",
				value="- if <code>auto</code> selected, will use <code>tol=0.5</code> if <code>model_type=alphafold2_multimer_v3</code> else <code>tol=0.0</code>.")
advanced_settings_gui.add_text(tag="relax_max_iterations",
				description="Relax max iterations (0=unlimited):",
				placeholder="200",
				value="200",
				remember_value=True)
advanced_settings_gui.add_HTML(tag="temp_output_adv_4",
				value="- max amber relax iterations, <code>0<\code> = unlimited (AlphaFold2 default, can take very long)")
advanced_settings_gui.add_dropdown(tag="pairing_strategy",
				description="Pairing strategy:",
				options=["greedy", "complete"],
				remember_value=True)
advanced_settings_gui.add_HTML(tag="temp_output_adv_5",
				value="- <code>greedy</code> = pair any taxonomically matching subsets, <code>complete</code> = all sequences have to match in one line.")
advanced_settings_gui.add_checkbox(tag="calc_extra_ptm",
				description="Calculate extra PTM scores",
				value=False,
				remember_value=True)
advanced_settings_gui.add_HTML(tag="temp_output_adv_6",
				value="- return pairwise chain iptm/actifptm<br><h4> Sample settings</h4> -  enable dropouts and increase number of seeds to sample predictions from uncertainty of the model.<br> -  decrease <code>max_msa</code> to increase uncertainity")
advanced_settings_gui.add_dropdown(tag="max_msa",
				description="Max MSA:",
				options=["auto", "512:1024", "256:512", "64:128", "32:64", "16:32"],
				remember_value=True)
advanced_settings_gui.add_dropdown(tag="num_seeds",
				description="Number of seeds:",
				options=[1,2,4,8,16],
				value=1,
				remember_value=True)
advanced_settings_gui.add_check(tag="use_dropout",
				description="Use dropout for sampling",
				value=False,
				remember_value=True)


advanced_settings_gui.add_HTML(tag="temp_output_adv_7",
				value="<h3>Save settings</h3>")
advanced_settings_gui.add_check(tag="save_all",
				description="Save all outputs",
				value=False,
				remember_value=True)
advanced_settings_gui.add_check(tag="save_recycles",
				description="Save recycles",
				value=False,
				remember_value=True)
advanced_settings_gui.add_check(tag="save_to_google_drive",
				description="Save results to Google Drive",
				value=False,
				remember_value=True)
advanced_settings_gui.add_HTML(tag="temp_output_adv_8",
				value="-  if the save_to_google_drive option was selected, the result zip will be uploaded to your Google Drive")
advanced_settings_gui.add_int_text(tag="dpi",
				description="DPI for images:",
				placeholder="200",
				value="200",
				remember_value=True)
advanced_settings_gui.add_HTML(tag="temp_output_adv_9",
				value="- set dpi for image resolution")

model_type = None
num_recycles = None
recycle_early_stop_tolerance = None
relax_max_iterations = None
pairing_strategy = None
calc_extra_ptm = None
max_msa = None
num_seeds = None
use_dropout = None
save_all = None
save_recycles = None
save_to_google_drive = None
dpi = None

def run_cell(_):
	global model_type, num_recycles, recycle_early_stop_tolerance
	global relax_max_iterations, pairing_strategy, calc_extra_ptm
	global max_msa, num_seeds, use_dropout
	global save_all, save_recycles, save_to_google_drive, dpi

	with advanced_settings_gui['output']:
		clear_output()
		model_type = advanced_settings_gui["model_type"].value
		num_recycles = advanced_settings_gui["num_recycles"].value
		recycle_early_stop_tolerance = advanced_settings_gui["recycle_early_stop_tolerance"].value
		relax_max_iterations = int(advanced_settings_gui["relax_max_iterations"].value)
		pairing_strategy = advanced_settings_gui["pairing_strategy"].value
		calc_extra_ptm = advanced_settings_gui["calc_extra_ptm"].value
		max_msa = advanced_settings_gui["max_msa"].value
		num_seeds = advanced_settings_gui["num_seeds"].value
		use_dropout = advanced_settings_gui["use_dropout"].value
		save_all = advanced_settings_gui["save_all"].value
		save_recycles = advanced_settings_gui["save_recycles"].value
		save_to_google_drive =  advanced_settings_gui["save_to_google_drive"].value
		dpi = int(advanced_settings_gui["dpi"].value)

		num_recycles = None if num_recycles == "auto" else int(num_recycles)
		recycle_early_stop_tolerance = None if recycle_early_stop_tolerance == "auto" else float(recycle_early_stop_tolerance)
		if max_msa == "auto": max_msa = None

		if save_to_google_drive:
			from pydrive2.drive import GoogleDrive
			from pydrive2.auth import GoogleAuth
			from google.colab import auth
			from oauth2client.client import GoogleCredentials
			auth.authenticate_user()
			gauth = GoogleAuth()
			gauth.credentials = GoogleCredentials.get_application_default()
			drive = GoogleDrive(gauth)
			print("You are logged into Google Drive and are good to go!")

		print("model_type",model_type)
		print("num_recycles",num_recycles)
		print("recycle_early_stop_tolerance",recycle_early_stop_tolerance)
		print("relax_max_iterations",relax_max_iterations)
		print("pairing_strategy",pairing_strategy)
		print("calc_extra_ptm",calc_extra_ptm)
		print("max_msa",max_msa)
		print("num_seeds",num_seeds)
		print("use_dropout",use_dropout)
		print("save_all",save_all)
		print("save_recycles",save_recycles)
		print("save_to_google_drive",save_to_google_drive)
		print("dpi",dpi)

advanced_settings_gui.add_callback("run_cell_adv", func=run_cell, values=advanced_settings_gui.elements, button_text="Submit")
advanced_settings_gui.add_output(tag="output")

# Display the GUI
advanced_settings_gui.show()
# Save settings
advanced_settings_gui._save_settings()

In [ ]:
#@title Run Prediction

import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from Bio import BiopythonDeprecationWarning
warnings.simplefilter(action='ignore', category=BiopythonDeprecationWarning)
from pathlib import Path
from colabfold.download import download_alphafold_params, default_data_dir
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run, set_model_type
from colabfold.plot import plot_msa_v2

from colabfold.colabfold import plot_protein
from pathlib import Path
import matplotlib.pyplot as plt

import os
import numpy as np

def input_features_callback(input_features):
  if display_images:
    plot_msa_v2(input_features)
    plt.show()
    plt.close()

def prediction_callback(protein_obj, length,
                        prediction_result, input_features, mode):
  model_name, relaxed = mode
  if not relaxed:
    if display_images:
      fig = plot_protein(protein_obj, Ls=length, dpi=150)
      plt.show()
      plt.close()

prediction_gui = EZInput(title="AlphaFold2_run_settings")

prediction_gui.add_HTML(tag="temp_output_run",
				value="<h3>Run Prediction</h3>")

prediction_gui.add_check(tag="display_images",
				description="Display images during prediction",
				value=True,
				remember_value=True)

results, queries, is_complex = None, None, None

def run_cell(_):
	global display_images, model_type, num_recycles, recycle_early_stop_tolerance
	global relax_max_iterations, pairing_strategy, calc_extra_ptm
	global max_msa, num_seeds, use_dropout
	global save_all, save_recycles, dpi, results, queries, is_complex

	with prediction_gui['output']:
		clear_output()
		display_images = prediction_gui["display_images"].value
		print("Display images during prediction:",display_images)

		try:
			K80_chk = os.popen('nvidia-smi | grep "Tesla K80" | wc -l').read()
		except:
			K80_chk = "0"
			pass
		if "1" in K80_chk:
			print("WARNING: found GPU Tesla K80: limited to total length < 1000")
			if "TF_FORCE_UNIFIED_MEMORY" in os.environ:
				del os.environ["TF_FORCE_UNIFIED_MEMORY"]
			if "XLA_PYTHON_CLIENT_MEM_FRACTION" in os.environ:
				del os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]

		# For some reason we need that to get pdbfixer to import
		if use_amber and f"/usr/local/lib/python{python_version}/site-packages/" not in sys.path:
			sys.path.insert(0, f"/usr/local/lib/python{python_version}/site-packages/")


		result_dir = jobname
		log_filename = os.path.join(jobname,"log.txt")
		setup_logging(Path(log_filename))

		queries, is_complex = get_queries(queries_path)
		model_type = set_model_type(is_complex, model_type)

		if "multimer" in model_type and max_msa is not None:
			use_cluster_profile = False
		else:
			use_cluster_profile = True

		download_alphafold_params(model_type, Path("."))
		results = run(
			queries=queries,
			result_dir=result_dir,
			use_templates=use_templates,
			custom_template_path=custom_template_path,
			num_relax=num_relax,
			msa_mode=msa_mode,
			model_type=model_type,
			num_models=5,
			num_recycles=num_recycles,
			relax_max_iterations=relax_max_iterations,
			recycle_early_stop_tolerance=recycle_early_stop_tolerance,
			num_seeds=num_seeds,
			use_dropout=use_dropout,
			model_order=[1,2,3,4,5],
			is_complex=is_complex,
			data_dir=Path("."),
			keep_existing_results=False,
			rank_by="auto",
			pair_mode=pair_mode,
			pairing_strategy=pairing_strategy,
			stop_at_score=float(100),
			prediction_callback=prediction_callback,
			dpi=dpi,
			zip_results=False,
			save_all=save_all,
			max_msa=max_msa,
			use_cluster_profile=use_cluster_profile,
			input_features_callback=input_features_callback,
			save_recycles=save_recycles,
			user_agent="colabfold/google-colab-main",
			calc_extra_ptm=calc_extra_ptm,
		)
		results_zip = f"{jobname}.result.zip"
		os.system(f"zip -r {results_zip} {jobname}")

prediction_gui.add_callback("run_cell_run", func=run_cell, values=prediction_gui.elements, button_text="Submit")
prediction_gui.add_output(tag="output")

# Display the prediction_gui
prediction_gui.show()
# Save settings
prediction_gui._save_settings()

In [ ]:
#@title Display 3D structure {run: "auto"}
import py3Dmol
import glob
import matplotlib.pyplot as plt
from colabfold.colabfold import plot_plddt_legend
from colabfold.colabfold import pymol_color_list, alphabet_list

display_3d_gui = EZInput("AlphaFold2_display_3D_settings")

display_3d_gui.add_dropdown(tag="rank_num",
				description="Select model rank to display:",
				options=["1", "2", "3", "4", "5"],
				value="1",
				remember_value=True)
display_3d_gui.add_dropdown(tag="color",
				description="Color scheme:",
				options=["chain", "lDDT", "rainbow"],
				value="lDDT",
				remember_value=True)
display_3d_gui.add_check(tag="show_sidechains",
				description="Show sidechains",
				value=False,
				remember_value=True)
display_3d_gui.add_check(tag="show_mainchains",
				description="Show mainchains",
				value=False,
				remember_value=True)

jobname_prefix = None

def run_cell(_):
	global rank_num, color, show_sidechains, show_mainchains, jobname_prefix

	with display_3d_gui['output']:
		clear_output()
		rank_num = int(display_3d_gui["rank_num"].value)
		color = display_3d_gui["color"].value
		show_sidechains = display_3d_gui["show_sidechains"].value
		show_mainchains = display_3d_gui["show_mainchains"].value

		print("Displaying model rank:",rank_num)
		print("Color scheme:",color)
		print("Show sidechains:",show_sidechains)
		print("Show mainchains:",show_mainchains)

		tag = results["rank"][0][rank_num - 1]
		jobname_prefix = ".custom" if msa_mode == "custom" else ""
		pdb_filename = f"{jobname}/{jobname}{jobname_prefix}_unrelaxed_{tag}.pdb"
		pdb_file = glob.glob(pdb_filename)

		def show_pdb(rank_num=1, show_sidechains=False, show_mainchains=False, color="lDDT"):
			model_name = f"rank_{rank_num}"
			view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js',)
			view.addModel(open(pdb_file[0],'r').read(),'pdb')

			if color == "lDDT":
				view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':50,'max':90}}})
			elif color == "rainbow":
				view.setStyle({'cartoon': {'color':'spectrum'}})
			elif color == "chain":
				chains = len(queries[0][1]) + 1 if is_complex else 1
				for n,chain,color in zip(range(chains),alphabet_list,pymol_color_list):
					view.setStyle({'chain':chain},{'cartoon': {'color':color}})

			if show_sidechains:
				BB = ['C','O','N']
				view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
									{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
				view.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
									{'sphere':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
				view.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
									{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
			if show_mainchains:
				BB = ['C','O','N','CA']
				view.addStyle({'atom':BB},{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})

			view.zoomTo()
			return view

		show_pdb(rank_num, show_sidechains, show_mainchains, color).show()
		if color == "lDDT":
			plot_plddt_legend().show()

display_3d_gui.add_callback("run_cell_run", func=run_cell, values=display_3d_gui.elements, button_text="Submit")
display_3d_gui.add_output(tag="output")

# Display the display_3d_gui
display_3d_gui.show()
# Save settings
display_3d_gui._save_settings()

In [ ]:
#@title Plots {run: "auto"}
from IPython.display import display, HTML
import base64
from html import escape

# see: https://stackoverflow.com/a/53688522
def image_to_data_url(filename):
  ext = filename.split('.')[-1]
  prefix = f'data:image/{ext};base64,'
  with open(filename, 'rb') as f:
    img = f.read()
  return prefix + base64.b64encode(img).decode('utf-8')

pae = ""
pae_file = os.path.join(jobname,f"{jobname}{jobname_prefix}_pae.png")
if os.path.isfile(pae_file):
    pae = image_to_data_url(pae_file)
cov = image_to_data_url(os.path.join(jobname,f"{jobname}{jobname_prefix}_coverage.png"))
plddt = image_to_data_url(os.path.join(jobname,f"{jobname}{jobname_prefix}_plddt.png"))
display(HTML(f"""
<style>
  img {{
    float:left;
  }}
  .full {{
    max-width:100%;
  }}
  .half {{
    max-width:50%;
  }}
  @media (max-width:640px) {{
    .half {{
      max-width:100%;
    }}
  }}
</style>
<div style="max-width:90%; padding:2em;">
  <h1>Plots for {escape(jobname)}</h1>
  { '<!--' if pae == '' else '' }<img src="{pae}" class="full" />{ '-->' if pae == '' else '' }
  <img src="{cov}" class="half" />
  <img src="{plddt}" class="half" />
</div>
"""))

In [ ]:
#@title Package and download results
#@markdown If you are having issues downloading the result archive, try disabling your adblocker and run this cell again. If that fails click on the little folder icon to the left, navigate to file: `jobname.result.zip`, right-click and select \"Download\" (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).

if msa_mode == "custom":
  print("Don't forget to cite your custom MSA generation method.")

files.download(f"{jobname}.result.zip")

if save_to_google_drive == True and drive:
  uploaded = drive.CreateFile({'title': f"{jobname}.result.zip"})
  uploaded.SetContentFile(f"{jobname}.result.zip")
  uploaded.Upload()
  print(f"Uploaded {jobname}.result.zip to Google Drive with ID {uploaded.get('id')}")

# Instructions <a name="Instructions"></a>
For detailed instructions, tips and tricks, see recently published paper at [Nature Protocols](https://www.nature.com/articles/s41596-024-01060-5)

**Quick start**
1. Paste your protein sequence(s) in the input field.
2. Press "Runtime" -> "Run all".
3. The pipeline consists of 5 steps. The currently running step is indicated by a circle with a stop sign next to it.

**Result zip file contents**

1. PDB formatted structures sorted by avg. pLDDT and complexes are sorted by pTMscore. (unrelaxed and relaxed if `use_amber` is enabled).
2. Plots of the model quality.
3. Plots of the MSA coverage.
4. Parameter log file.
5. A3M formatted input MSA.
6. A `predicted_aligned_error_v1.json` using [AlphaFold-DB's format](https://alphafold.ebi.ac.uk/faq#faq-7) and a `scores.json` for each model which contains an array (list of lists) for PAE, a list with the average pLDDT and the pTMscore.
7. BibTeX file with citations for all used tools and databases.

At the end of the job a download modal box will pop up with a `jobname.result.zip` file. Additionally, if the `save_to_google_drive` option was selected, the `jobname.result.zip` will be uploaded to your Google Drive.

**MSA generation for complexes**

For the complex prediction we use unpaired and paired MSAs. Unpaired MSA is generated the same way as for the protein structures prediction by searching the UniRef100 and environmental sequences three iterations each.

The paired MSA is generated by searching the UniRef100 database and pairing the best hits sharing the same NCBI taxonomic identifier (=species or sub-species). We only pair sequences if all of the query sequences are present for the respective taxonomic identifier.

**Using a custom MSA as input**

To predict the structure with a custom MSA (A3M formatted): (1) Change the `msa_mode`: to "custom", (2) Wait for an upload box to appear at the end of the "MSA options ..." box. Upload your A3M. The first fasta entry of the A3M must be the query sequence without gaps.

It is also possilbe to provide custom MSAs for complex predictions. Read more about the format [here](https://github.com/sokrypton/ColabFold/issues/76).

As an alternative for MSA generation the [HHblits Toolkit server](https://toolkit.tuebingen.mpg.de/tools/hhblits) can be used. After submitting your query, click "Query Template MSA" -> "Download Full A3M". Download the A3M file and upload it in this notebook.

**PDB100** <a name="pdb100"></a>

As of 23/06/08, we have transitioned from using the PDB70 to a 100% clustered PDB, the PDB100. The construction methodology of PDB100 differs from that of PDB70.

The PDB70 was constructed by running each PDB70 representative sequence through [HHblits](https://github.com/soedinglab/hh-suite) against the [Uniclust30](https://uniclust.mmseqs.com/). On the other hand, the PDB100 is built by searching each PDB100 representative structure with [Foldseek](https://github.com/steineggerlab/foldseek) against the [AlphaFold Database](https://alphafold.ebi.ac.uk).

To maintain compatibility with older Notebook versions and local installations, the generated files and API responses will continue to be named "PDB70", even though we're now using the PDB100.

**Using custom templates** <a name="custom_templates"></a>

To predict the structure with a custom template (PDB or mmCIF formatted): (1) change the `template_mode` to "custom" in the execute cell and (2) wait for an upload box to appear at the end of the "Input Protein" box. Select and upload your templates (multiple choices are possible).

* Templates must follow the four letter PDB naming with lower case letters.

* Templates in mmCIF format must contain `_entity_poly_seq`. An error is thrown if this field is not present. The field `_pdbx_audit_revision_history.revision_date` is automatically generated if it is not present.

* Templates in PDB format are automatically converted to the mmCIF format. `_entity_poly_seq` and `_pdbx_audit_revision_history.revision_date` are automatically generated.

If you encounter problems, please report them to this [issue](https://github.com/sokrypton/ColabFold/issues/177).

**Comparison to the full AlphaFold2 and AlphaFold2 Colab**

This notebook replaces the homology detection and MSA pairing of AlphaFold2 with MMseqs2. For a comparison against the [AlphaFold2 Colab](https://colab.research.google.com/github/deepmind/alphafold/blob/main/notebooks/AlphaFold.ipynb) and the full [AlphaFold2](https://github.com/deepmind/alphafold) system read our [paper](https://www.nature.com/articles/s41592-022-01488-1).

**Troubleshooting**
* Check that the runtime type is set to GPU at "Runtime" -> "Change runtime type".
* Try to restart the session "Runtime" -> "Factory reset runtime".
* Check your input sequence.

**Known issues**
* Google Colab assigns different types of GPUs with varying amount of memory. Some might not have enough memory to predict the structure for a long sequence.
* Your browser can block the pop-up for downloading the result file. You can choose the `save_to_google_drive` option to upload to Google Drive instead or manually download the result file: Click on the little folder icon to the left, navigate to file: `jobname.result.zip`, right-click and select \"Download\" (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).

**Limitations**
* Computing resources: Our MMseqs2 API can handle ~20-50k requests per day.
* MSAs: MMseqs2 is very precise and sensitive but might find less hits compared to HHblits/HMMer searched against BFD or MGnify.
* We recommend to additionally use the full [AlphaFold2 pipeline](https://github.com/deepmind/alphafold).

**Description of the plots**
*   **Number of sequences per position** - We want to see at least 30 sequences per position, for best performance, ideally 100 sequences.
*   **Predicted lDDT per position** - model confidence (out of 100) at each position. The higher the better.
*   **Predicted Alignment Error** - For homooligomers, this could be a useful metric to assess how confident the model is about the interface. The lower the better.

**Bugs**
- If you encounter any bugs, please report the issue to https://github.com/sokrypton/ColabFold/issues

**License**

The source code of ColabFold is licensed under [MIT](https://raw.githubusercontent.com/sokrypton/ColabFold/main/LICENSE). Additionally, this notebook uses the AlphaFold2 source code and its parameters licensed under [Apache 2.0](https://raw.githubusercontent.com/deepmind/alphafold/main/LICENSE) and [CC BY 4.0](https://creativecommons.org/licenses/by-sa/4.0/) respectively. Read more about the AlphaFold license [here](https://github.com/deepmind/alphafold).

**Acknowledgments**
- We thank the AlphaFold team for developing an excellent model and open sourcing the software.

- [KOBIC](https://kobic.re.kr) and [Söding Lab](https://www.mpinat.mpg.de/soeding) for providing the computational resources for the MMseqs2 MSA server.

- Richard Evans for helping to benchmark the ColabFold's Alphafold-multimer support.

- [David Koes](https://github.com/dkoes) for his awesome [py3Dmol](https://3dmol.csb.pitt.edu/) plugin, without whom these notebooks would be quite boring!

- Do-Yoon Kim for creating the ColabFold logo.

- A colab by Sergey Ovchinnikov ([@sokrypton](https://twitter.com/sokrypton)), Milot Mirdita ([@milot_mirdita](https://twitter.com/milot_mirdita)) and Martin Steinegger ([@thesteinegger](https://twitter.com/thesteinegger)).
